# CS 123 — How to Train Your Pupper 🐶

Train a walking policy for **Pupper v3** in massively parallel GPU simulation,
deploy it on your robot with one command.

**The catch:** every task ships with all reward weights at **zero**. Untouched, the
robot learns to do nothing, beautifully. What to reward, what to penalize, and by
how much is your job: edit the ✏️ config blocks, train, watch the metrics,
iterate. Everything happens in this notebook — no repo editing needed for the
main lab.

*Runtime → Change runtime type → **G4 GPU** (an RTX Pro 6000 — a 3,000-iteration
run finishes in about 18 minutes on it), then run cells top to bottom.*

**Authors:** Kevin Zakka for mjlab, the training infrastructure; JC for Pupper v3 integration & modifications for the course. Hope you enjoy this new version!

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/cs123-stanford/pupper-mjlab.git"
BRANCH = "main"

import os

!nvidia-smi -L
if not os.path.exists("/content/pupper-mjlab"):
  !git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/pupper-mjlab
%cd /content/pupper-mjlab
!pip install -q -e .

# The running kernel can't see an editable install's path hooks; wire src/
# onto the path so imports work without a runtime restart.
import sys

sys.path.insert(0, "/content/pupper-mjlab/src")
for _m in [k for k in list(sys.modules) if k == "mjlab" or k.startswith("mjlab.")]:
  del sys.modules[_m]
import mjlab.envs  # smoke-test

print("mjlab ready")

## 2. Log into Weights & Biases

Everything lands in [W&B](https://wandb.ai): reward curves, the tracking-error
metrics that tell you the truth, and your deployable `policy.json` (uploaded
automatically every 50 iterations and when training stops). Free account.

Grab your API key from [wandb.ai/authorize](https://wandb.ai/authorize) and paste
it into the cell below **once** — it then survives Colab reconnects, no
re-authentication dance every time the runtime blinks.

In [ ]:
WANDB_API_KEY = ""  # paste your key from https://wandb.ai/authorize between the quotes

import os

import wandb

if WANDB_API_KEY:
  os.environ["WANDB_API_KEY"] = WANDB_API_KEY
  wandb.login(key=WANDB_API_KEY, relogin=True)
else:
  wandb.login()  # no key pasted: falls back to the interactive prompt

## 3. Choose your task

| Task | What it is |
| --- | --- |
| `Mjlab-VelocityFS-Flat-Pupper-v3` | Velocity tracking **from scratch** — no reference motion, the policy invents its own gait. Start here. |
| `Mjlab-StableGait-Flat-Pupper-v3` | The trot: a triangular gait for fore/aft, stepping-in-place for turn/side-step. |
| `Mjlab-StableGait-Bumpy-Pupper-v3` | The trot on rough terrain — the sim2real robustness pass. |

VelocityFS needs nothing extra — start there. **StableGait needs your lift gait
first**: design it in section 4 below.

(The registry holds a few more tasks. Explore at your own risk — that road leads
into the repo itself, and Claude Code makes a good trail guide.)

In [ ]:
TASK = "Mjlab-VelocityFS-Flat-Pupper-v3"
NUM_ENVS = 4096  # halve it if you hit GPU OOM; 5/96 GB VRAM for rtxpro6000
ITERATIONS = 3000  # walking usually shows up well before this

## 4. Design the lift gait ✏️ (StableGait only)

StableGait picks its reference from the command: the shipped trot plays for
forward and backward walking, and a second gait — **lift** — plays whenever the
command is a pure turn or sidestep. Its job is to keep the feet stepping (so the
policy can rotate the body between touchdowns) without the reference itself
dragging the robot anywhere.

You have built this machinery before: the reference generator is your lab 3 code
— the triangle keyframe interpolation, the gradient-descent IK, the Raibert
stance/swing cycle — with the trot as the worked example. Before you fill in the
four numbers, decide (and write down in your lab doc):

- **Stride** — the trot slides each planted foot backward so the body moves
  forward. What must the stride be if the reference must not oppose a
  turn-in-place?
- **Touchdown phases** — which legs should land together? Does the pairing you
  picked in lab 3 for forward trotting still make sense when the robot is not
  going anywhere?
- **Stance depth / swing lift** — the trot's values hold the body at standing
  height and clear the terrain. Any reason to change them?

The cell writes your design into the repo *and* the running kernel, so training
and the visualizer both see it. **Look at it before you train with it** —
interrupt the cell to stop the viewer:

```python
!cd /content/pupper-mjlab && python -m mjlab.tasks.pupper_gait.visualize_reference --gait lift --viser --share
```

In [ ]:
# Only the StableGait tasks read this — VelocityFS ignores it entirely.
LIFT_TOUCHDOWN = None  # (FR, FL, BR, LB) touchdown phases in [0, 1); the trot's is (0.0, 0.5, 0.5, 0.0)
LIFT_STRIDE = None  # fore-aft half-stride [m]; trot: 0.05
LIFT_STANCE_Z = None  # foot depth below the hip during stance [m]; trot: -0.14
LIFT_SWING_LIFT = None  # foot rise above the stance plane in swing [m]; trot: 0.09

# Apply: writes the values into the repo file (for the visualizer and fresh
# kernels) and into this kernel (for training, no restart). Re-run after every
# change.
import re

from mjlab.tasks.pupper_gait.mdp import gait_reference

_lift = {
  "_GAIT_TOUCHDOWN": LIFT_TOUCHDOWN,
  "_GAIT_STRIDE": LIFT_STRIDE,
  "_GAIT_STANCE_Z": LIFT_STANCE_Z,
  "_GAIT_SWING_LIFT": LIFT_SWING_LIFT,
}
if all(v is None for v in _lift.values()):
  print("Lift gait not designed yet — fine for VelocityFS, blocks StableGait.")
else:
  _path = "/content/pupper-mjlab/src/mjlab/tasks/pupper_gait/mdp/gait_reference.py"
  _text = open(_path).read()
  for _name, _value in _lift.items():
    getattr(gait_reference, _name)["lift"] = _value
    _text = re.sub(
      rf'^{_name}\["lift"\] = .*?(  #.*)?$',
      lambda m, n=_name, v=_value: f'{n}["lift"] = {v!r}{m.group(1) or ""}',
      _text,
      count=1,
      flags=re.M,
    )
  open(_path, "w").write(_text)
  for _key in [k for k in gait_reference._TABLE_CACHE if k[1] == "lift"]:
    del gait_reference._TABLE_CACHE[_key]
  # Runs the IK now, so a bad design fails here and not mid-training.
  _table = gait_reference.build_joint_reference_table(100, "lift")
  print(f"Lift gait applied (table {_table.shape}) — repo and kernel both updated.")

## 5. Reward weights ✏️

Positive encourages, negative penalizes, zero doesn't pay.

How the numbers behave: the **positive** terms are shaped `exp(-error²/std²)`,
bounded in `[0, 1]` per step — so a positive weight is exactly the *most* that
term can pay per step, and positive terms compete with each other by ratio.
The **negative** penalties multiply raw, unbounded physical quantities
(torque², rad/s², …), so their useful magnitudes vary wildly between terms —
set them by trial and error, and watch each term's `Episode_Reward/...` curve
on W&B to see what it actually costs.

Judge runs by `Metrics/twist/error_vel_xy` and `error_vel_yaw` on W&B (tracking
error, lower = better) — **not** by the reward curve. A rising reward with flat
errors means the policy found a way to farm your weights without walking. It will.

In [ ]:
REWARD_WEIGHTS = {
  # objectives
  "track_linear_velocity": 0.0,  # follow commanded vx/vy
  "track_yaw_velocity": 0.0,  # follow commanded turn rate
  "upright": 0.0,  # stay upright
  "air_time": 0.0,  # take real steps
  "base_height": 0.0,  # hold standing height
  "pose": 0.0,  # sensible posture (VelocityFS only)
  "gait_tracking": 0.0,  # match the reference gait (StableGait only)
  "track_angular_velocity": 0.0,  # roll/pitch-rate stabilizer (StableGait only)
  # penalties (negative weights)
  "termination": 0.0,  # falling over
  "orientation_l2": 0.0,  # body tilt
  "lin_vel_z_l2": 0.0,  # vertical bouncing
  "ang_vel_xy_l2": 0.0,  # roll/pitch thrash
  "foot_slip": 0.0,  # planted feet sliding
  "action_rate_l2": 0.0,  # jerky actions (this is what shakes real robots)
  "joint_torques_l2": 0.0,  # effort
  "joint_acc_l2": 0.0,  # joint acceleration
  "stand_still_pose": 0.0,  # fidgeting at zero command
  "stand_still_joint_velocity": 0.0,  # creeping at zero command
  "knee_ground_contact": 0.0,  # knees on the floor
  "abduction_angle": 0.0,  # splayed legs
  "self_collision_l": 0.0,  # left legs colliding
  "self_collision_r": 0.0,  # right legs colliding
}

## 6. Domain randomization ✏️

The simulator is not your robot: real motors run weaker or stronger than modeled,
real floors grip differently. Each `(low, high)` range is sampled per episode —
too narrow breaks on hardware, too wide trains a timid crouch.

In [ ]:
KP_MULTIPLIER_RANGE = (0.6, 1.1)  # motor position-gain multiplier (1.0 = as modeled)
KD_MULTIPLIER_RANGE = (0.8, 1.5)  # motor damping-gain multiplier
FRICTION_RANGE = (0.6, 1.4)  # foot-ground friction coefficient

## 7. Train 🚀

Blocks while it runs (~18 min for a 3,000-iteration run on the G4 / RTX Pro
6000 runtime) — open the
printed W&B link to watch live. **The stop button is safe**: interrupting exports
and uploads the current policy, and `policy.json` refreshes on W&B every 50
iterations regardless. (W&B shows the *first* upload's timestamp; the content is
current.)

StableGait note: the first ~500 iterations run airborne, learning the reference
gait with only `gait_tracking` paying, before gravity drops in.

### Watch it live 👀 (optional but recommended — run *before* the training cell)

Launches a background viewer on this same GPU that follows the run you start
below. Open the printed link in any browser tab — the robot appears once the
first checkpoint lands, about a minute into training (if the page looks empty,
reload it). In the viewer, **Checkpoints → Sync → Use Latest** hot-loads the newest
policy as training writes them; the sliders drive the robot as usual.

**The GPU cost is the open tab, not this cell**: with no tab connected the
viewer pauses itself and training runs at full speed (measured 0% overhead);
while a tab is connected, training runs ~25–30% slower. So peek, then close
the tab — don't leave it open in the background. Re-run this cell before each
new training run so the viewer follows the right one.


In [ ]:
import subprocess
import sys
import time
from pathlib import Path

URL_FILE = Path("/content/viewer_url.txt")
LOG_FILE = Path("/content/viewer.log")
PID_FILE = Path("/content/viewer_pid.txt")

# Restart the watcher so it latches onto the run started next, not an old one.
if PID_FILE.exists():
  subprocess.run(["kill", PID_FILE.read_text().strip()], capture_output=True)
URL_FILE.unlink(missing_ok=True)
proc = subprocess.Popen(
  [sys.executable, "notebooks/watch_live.py", TASK, "--url-file", str(URL_FILE)],
  stdout=LOG_FILE.open("w"),
  stderr=subprocess.STDOUT,
)
PID_FILE.write_text(str(proc.pid))

for _ in range(45):
  if URL_FILE.exists():
    print("Watch live here:", URL_FILE.read_text())
    break
  time.sleep(2)
else:
  print("Viewer didn't come up -- check", LOG_FILE)
print("Robot appears once the first checkpoint lands (~1 min into training).")

In [ ]:
from mjlab.scripts.train import TrainConfig, launch_training
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg
import mjlab.tasks  # register tasks

env_cfg = load_env_cfg(TASK)
for name, weight in REWARD_WEIGHTS.items():
  if name in env_cfg.rewards:  # terms a task doesn't have are skipped
    env_cfg.rewards[name].weight = weight
env_cfg.events["pd_gains"].params["kp_range"] = KP_MULTIPLIER_RANGE
env_cfg.events["pd_gains"].params["kd_range"] = KD_MULTIPLIER_RANGE
env_cfg.events["foot_friction"].params["ranges"] = FRICTION_RANGE
env_cfg.scene.num_envs = NUM_ENVS

agent_cfg = load_rl_cfg(TASK)
agent_cfg.max_iterations = ITERATIONS
agent_cfg.logger = "wandb"

launch_training(TASK, TrainConfig(env=env_cfg, agent=agent_cfg))

## 8. Watch it 🕹️

Loads your newest checkpoint into an interactive 3D viewer (public link, works
from Colab). Drive the robot with the sliders.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import viser
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
from mjlab.viewer import ViserPlayViewer

runs = sorted(Path("logs/rsl_rl").glob("*/*"), key=lambda p: p.stat().st_mtime)
assert runs, "No runs yet -- train first."
ckpts = sorted(runs[-1].glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
print("Using checkpoint:", ckpts[-1])

device = "cuda:0"
env_cfg = load_env_cfg(TASK, play=True)
env_cfg.scene.num_envs = 1
agent_cfg = load_rl_cfg(TASK)
env = RslRlVecEnvWrapper(
  ManagerBasedRlEnv(cfg=env_cfg, device=device), clip_actions=agent_cfg.clip_actions
)
runner_cls = load_runner_cls(TASK) or MjlabOnPolicyRunner
runner = runner_cls(env, asdict(agent_cfg), device=device)
runner.load(str(ckpts[-1]), load_cfg={"actor": True}, strict=True, map_location=device)

server = viser.ViserServer(label="pupper", share=True)
print("Open the viewer here:", server.request_share_url())
ViserPlayViewer(env, runner.get_inference_policy(device=device), viser_server=server).run()

## 9. Deploy 🤖

Your `policy.json` is already on W&B. Make sure you log into the same W&B account on Pupper: 
```bash
wandb login
```

Then, on the robot:

```bash
cd ~/pupper_gait_deploy
./deploy.sh mjlab/<your-run-id>   # run id = last part of your W&B run URL
```

Press **square** on the gamepad to activate *your* policy — the other buttons keep
the reference policies for comparison:

![CS 123 gamepad gait diagram](https://raw.githubusercontent.com/cs123-stanford/pupper_gait_deploy/main/docs/gamepad_gait_diagram.png)

Iterate: tweak the ✏️ blocks → retrain → `./deploy.sh` again. And when you're
done: **Runtime → Disconnect** so the GPU goes back in the pool.

---
### When it doesn't walk (it won't, at first)
- **Stands still forever** — tracking terms don't out-pay standing. Raise
  `track_linear_velocity` until they do; then check yaw didn't get abandoned.
- **Reward rises, error metrics don't** — some term is being farmed. Find it on
  W&B and cut it down.
- **Robot terminates instantly every run** — one or a few penalty weights are too high. Find them on
  W&B and cut them down.
- **Walks in sim, shakes on the robot** — `action_rate_l2` and the DR ranges are
  what transfer.
- **Vibrates / bounces** — `lin_vel_z_l2` and `ang_vel_xy_l2` are too cheap.
- **StableGait refuses to build** — you haven't designed the lift gait yet; go
  back to section 4 and run the apply cell.
- **StableGait ignores the trot** — `gait_tracking` was zero during the airborne
  phase, so nothing was learned before gravity arrived.
- **StableGait fights every turn** — look at what your lift gait asks the feet to
  do during a turn-in-place. The visualizer command in section 4 shows you.

Have fun training Pupper! If you want to train Pupper for cooler tasks, fork from pupper-mjlab to make your own editable codebase, and explore with Claude Code/Codex: the repo include a few more hidden tasks, which will be covered in the optional lab. They will be fantastic starting points for your final project.